# OneRef — notebook autonome
Tout le code du projet (`src/inspector`) est dans ce fichier : les cellules `%%writefile` recréent le package `inspector/` sur le disque du noyau, puis on lance le pipeline.

Ordre : **1** installation → **2** variables d'environnement → **3** code du package → **4** tests → **5** exécution.
GPU recommandé : L4 (bf16). Le split `test` ne se lance qu'une fois, avec `--frozen-config`.

## 1. Installation

In [ ]:
!pip install -q pydantic pydantic-settings "pydantic-ai-slim[openai]" "transformers>=4.56" scipy pillow pytest

In [ ]:
# Seulement si PROVIDER = "vllm" (GPU Linux/Colab). Peut remplacer torch : redémarre le noyau si demandé.
# !pip install -q vllm

## 2. Configuration
Même méthode que PasteurAIze : tout est centralisé dans `Settings` (préfixe `ONEREF_`), puis chaque agent reçoit un `OpenAIChatModel(model_name=..., provider=OpenAIProvider(api_key=..., base_url=...))` (`make_model` dans `inspector/agents/llm.py`). Choisis `PROVIDER` : **groq** (Qwen sur Groq, gratuit ; repli automatique sur Gemini quand le quota est épuisé), **gemini** (gratuit, clé Google), **router** (routeur Hugging Face, crédits payants) ou **vllm** (serveur local par passe, GPU). L'expert visuel DINOv3 tourne toujours en local.

In [ ]:
import os

# --- 1. Fournisseur des LLM : "groq" (Qwen sur Groq, backup Gemini) | "gemini" | "router" (HF, payant) | "vllm" (local/Colab, GPU) ---
PROVIDER = "groq"

# --- 2. Clés (ou variables d'environnement / secrets Colab). Ne jamais commiter de vraie clé. ---
HF_TOKEN       = os.environ.get("HF_TOKEN", "")          # DINOv3 (licence Meta à accepter) + routeur HF + téléchargement MMAD
GROQ_API_KEY   = os.environ.get("GROQ_API_KEY", "")      # https://console.groq.com/keys  (PROVIDER = "groq")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")    # https://aistudio.google.com/apikey (PROVIDER = "gemini", ou backup de "groq")
# Colab :  from google.colab import userdata ; GROQ_API_KEY = userdata.get("GROQ_API_KEY") ; ...

# --- 3. MMAD : "sample" (mmad.json + un petit zip, pour tester) | "full" (~28 Go) | "none" (déjà présent) ---
MMAD_MODE = "sample"
SAMPLE_ZIPS = ["DS-MVTec.zip"]        # DS-MVTec 1.6 Go, VisA 1.9 Go, MVTec-AD 5.3 Go, MVTec-LOCO 6.1 Go, GoodsAD 13.3 Go
MMAD_ROOT = "data/MMAD"               # mettre un autre disque si C: est plein

GEMINI = {
    "ONEREF_LLM_BASE_URL": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "ONEREF_LLM_API_KEY": GEMINI_API_KEY,
    "ONEREF_READER_A_MODEL": "gemini-3.8-flash",
    "ONEREF_READER_B_MODEL": "gemini-3.5-flash-lite",
    "ONEREF_CRITIC_MODEL": "gemini-3.8-flash",
}
PROFILES = {
    # Groq en premier ; si Groq renvoie une erreur (quota 429...), repli automatique sur Gemini (FallbackModel) si une clé Gemini est fournie
    "groq": {
        "ONEREF_USE_GROK": "true",
        "ONEREF_GROK_API_KEY": GROQ_API_KEY,
        "ONEREF_GROK_BASE_URL": "https://api.groq.com/openai/v1",
        "ONEREF_GROK_READER_A_MODEL": "qwen/qwen3.8-27b",
        "ONEREF_GROK_READER_B_MODEL": "qwen/qwen3.8-27b",
        "ONEREF_GROK_CRITIC_MODEL": "qwen/qwen3.8-27b",
        **(GEMINI if GEMINI_API_KEY else {"ONEREF_BACKUP_ENABLED": "false"}),    # sans clé Gemini : pas de backup
    },
    "gemini": GEMINI,
    "router": {
        "ONEREF_LLM_BASE_URL": "https://router.huggingface.co/v1",
        "ONEREF_LLM_API_KEY": HF_TOKEN,
        "ONEREF_READER_A_MODEL": "Qwen/Qwen3-VL-30B-A3B-Instruct:novita",
        "ONEREF_READER_B_MODEL": "Qwen/Qwen3.8-27B:novita",
        "ONEREF_CRITIC_MODEL": "Qwen/Qwen3-VL-30B-A3B-Instruct:novita",
    },
    "vllm": {   # un `vllm serve` par passe (GPU requis, ex. Colab L4) : protocole strict, modèles <= 8B
        "ONEREF_LLM_PROVIDER": "vllm",
        "ONEREF_LLM_BASE_URL": "http://localhost:8000/v1",
        "ONEREF_LLM_API_KEY": "EMPTY",
        "ONEREF_VLLM_AUTOSTART": "true",
        "ONEREF_READER_A_MODEL": "Qwen/Qwen3-VL-8B-Instruct",
        "ONEREF_READER_B_MODEL": "OpenGVLab/InternVL3-8B",
        "ONEREF_CRITIC_MODEL": "Qwen/Qwen3-VL-8B-Instruct",
    },
}
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.update(PROFILES[PROVIDER])      # les variables d'environnement l'emportent sur un éventuel .env
os.environ.update({
    "ONEREF_MMAD_ROOT": MMAD_ROOT,
    "ONEREF_MMAD_JSON": f"{MMAD_ROOT}/mmad.json",
    "ONEREF_OUTPUTS_DIR": "outputs",
    "ONEREF_SPLITS_DIR": "data/splits",
    "ONEREF_DEVICE": "cuda",   # expert DINOv3 ; "cpu" sans GPU
    "PYTHONPATH": os.getcwd(),
})
OPT = "--available-only" if MMAD_MODE == "sample" else ""   # n'utiliser que les images téléchargées
print("provider:", PROVIDER, "| groq:", bool(GROQ_API_KEY), "| gemini (backup):", bool(GEMINI_API_KEY), "| HF_TOKEN:", bool(HF_TOKEN), "| MMAD:", MMAD_MODE)

### Télécharger MMAD (échantillon ou complet)
`MMAD_MODE = "sample"` : `mmad.json` + un seul zip (par défaut DS-MVTec, 1,6 Go), puis le pipeline ne garde que les images présentes (`--available-only`). `"full"` : les 5 zips (~28 Go). Le zip est supprimé après extraction. Fonctionne sous Windows, Linux et Colab.

In [ ]:
%%writefile download_mmad.py
"""Download MMAD from Hugging Face (jiang-cc/MMAD): a small sample, or everything.

    python scripts/download_mmad.py --sample                 # mmad.json + DS-MVTec.zip (~1.6 GB)
    python scripts/download_mmad.py --sample --zips VisA.zip
    python scripts/download_mmad.py --full                  # all zips (~28 GB)
    python scripts/download_mmad.py --sample --dest D:/MMAD # then ONEREF_MMAD_ROOT=D:/MMAD

Then run the pipeline with ``--available-only`` so that only the images that exist are used.
Needs HF_TOKEN (env) if the dataset is gated. Zips are deleted after extraction (--keep-zips to keep).
Zip sizes: DS-MVTec 1.6 GB, VisA 1.9 GB, MVTec-AD 5.3 GB, MVTec-LOCO 6.1 GB, GoodsAD 13.3 GB.
"""
import argparse
import os
import zipfile
from pathlib import Path

from huggingface_hub import hf_hub_download

REPO = "jiang-cc/MMAD"
ALL_ZIPS = ["DS-MVTec.zip", "VisA.zip", "MVTec-AD.zip", "MVTec-LOCO.zip", "GoodsAD.zip"]


def download_mmad(dest: str | Path = "data/MMAD", zips: list[str] | None = None, keep_zips: bool = False) -> Path:
    """Fetch ``mmad.json`` and the given zips (default: the smallest, DS-MVTec) into ``dest``."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    token = os.environ.get("HF_TOKEN") or None
    hf_hub_download(REPO, "mmad.json", repo_type="dataset", local_dir=dest, token=token)
    for name in zips or ["DS-MVTec.zip"]:
        z = Path(hf_hub_download(REPO, name, repo_type="dataset", local_dir=dest, token=token))
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        if not keep_zips:
            z.unlink()
        print(f"{name}: extracted to {dest}")
    return dest


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--sample", action="store_true", help="mmad.json + one small zip (default DS-MVTec.zip)")
    g.add_argument("--full", action="store_true", help="mmad.json + all zips (~28 GB)")
    ap.add_argument("--zips", nargs="+", choices=ALL_ZIPS, help="zips for --sample")
    ap.add_argument("--dest", default=os.environ.get("ONEREF_MMAD_ROOT", "data/MMAD"))
    ap.add_argument("--keep-zips", action="store_true")
    a = ap.parse_args()
    download_mmad(a.dest, ALL_ZIPS if a.full else a.zips, a.keep_zips)

In [ ]:
if MMAD_MODE in ("sample", "full"):
    from download_mmad import ALL_ZIPS, download_mmad
    download_mmad(MMAD_ROOT, ALL_ZIPS if MMAD_MODE == "full" else SAMPLE_ZIPS)
else:
    print("MMAD_MODE = none : rien à télécharger")

## 3. Code du package `inspector`

In [ ]:
import os
for d in ['inspector/agents','inspector/config','inspector/orchestrator','inspector/schemas','inspector/utils','tests']:
    os.makedirs(d, exist_ok=True)
    if d != 'tests': open(f'{d}/__init__.py','w').close()
open('inspector/__init__.py','w').close()

In [ ]:
%%writefile inspector/agents/aggregator.py
"""Aggregator: weighted vote per question.

* Each reader votes for its option with weight ``Settings.weights[agent][subtask]`` (default 1.0),
  fixed on the dev split.
* On anomaly discrimination the visual expert votes too (defect if score >= threshold).
* Ties: higher summed confidence, then alphabetical letter (deterministic).
* No valid vote at all -> ``selected_option=None`` (scored as an error).
"""
from __future__ import annotations

from collections import defaultdict
from typing import Any

from inspector.agents.base import BaseAgent
from inspector.agents.critic_verifier import says_no_defect, says_yes_defect
from inspector.config.settings import Settings
from inspector.schemas.agent_outputs import FinalAnswer
from inspector.schemas.mmad import SubTask
from inspector.schemas.state import InspectionState


class Aggregator(BaseAgent):
    name = "aggregator"

    def __init__(self, settings: Settings):
        self.s = settings

    def _w(self, agent: str, subtask: SubTask) -> float:
        return self.s.weights.get(agent, {}).get(subtask.value, self.s.default_weight)

    def _run(self, state: InspectionState, rec: dict[str, Any]) -> InspectionState:
        ev = state.visual_evidence
        for q in state.questions:
            votes: dict[str, float] = defaultdict(float)
            conf: dict[str, float] = defaultdict(float)
            for agent, store in (("reader_a", state.reader_a_responses), ("reader_b", state.reader_b_responses)):
                a = store.get(q.id)
                if a is not None:
                    votes[a.selected_option] += self._w(agent, q.subtask)
                    conf[a.selected_option] += a.confidence
            if ev is not None and q.subtask == SubTask.ANOMALY_DISCRIMINATION:
                flag = ev.confidence_score >= self.s.expert_flag_threshold
                for letter in q.options:
                    if (flag and says_yes_defect(q, letter)) or (not flag and says_no_defect(q, letter)):
                        votes[letter] += self.s.expert_weight_discrimination
            best = min(votes, key=lambda k: (-votes[k], -conf[k], k)) if votes else None
            state.final_answers[q.id] = FinalAnswer(question_id=q.id, selected_option=best, votes=dict(votes))
        rec["final"] = {k: v.selected_option for k, v in state.final_answers.items()}
        return state

In [ ]:
%%writefile inspector/agents/base.py
"""Base class for all agents.

Lifecycle (driven by ``orchestrator.pass_runner``): ``load()`` once per pass, ``run()`` for
every image of the split, ``unload()`` to free the GPU before the next pass. An agent never
keeps weights after ``unload()``, which is what keeps peak VRAM at "one model at a time".
"""
from __future__ import annotations

import gc
from abc import ABC, abstractmethod
from typing import Any

from inspector.schemas.state import InspectionState
from inspector.utils.telemetry import Telemetry


def free_gpu() -> None:
    """Release Python garbage and cached CUDA blocks (call after dropping model refs)."""
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    except Exception:  # noqa: BLE001 - torch is optional for CPU-only / mock runs
        pass


class BaseAgent(ABC):
    name: str = "agent"

    def load(self) -> None:
        """Load weights. Default: nothing to load (rule-based agents)."""

    def unload(self) -> None:
        """Drop weights and free VRAM. Subclasses must null their model refs first."""
        free_gpu()

    def applies(self, state: InspectionState) -> bool:
        """False => the runner skips this image for this pass (and may skip loading the model)."""
        return True

    def run(self, state: InspectionState, tel: Telemetry) -> InspectionState:
        """Run on one image inside a telemetry span; returns the updated state."""
        with tel.span(self.name, state.image_id) as rec:
            state = self._run(state, rec)
        return state

    @abstractmethod
    def _run(self, state: InspectionState, rec: dict[str, Any]) -> InspectionState:
        """Do the work; put prompts / raw outputs / token counts in ``rec``."""

In [ ]:
%%writefile inspector/agents/critic_verifier.py
"""Critic / verifier (service agent): deterministic rules + an optional LLM critic (pydantic-ai).

Rules (always on, no model), in order:
1. Schema/format: a reader answer is missing (invalid after retry) -> ``is_valid=False``, reopen it.
2. Consistency across questions of the same image (per reader):
   a. discrimination says "No defect" but the expert sees a strong anomaly (score >= threshold);
   b. a localization answer names a 3x3 cell different from the expert's region while the expert is confident;
   c. (optional, off by default) discrimination says "No" yet defect-localization questions exist.
      Disabled by default: question presence reveals the label in MMAD (strict protocol).
3. Disagreement between readers is NOT a conflict by itself: the aggregator votes.

LLM critic (``use_llm_critic``, Qwen3-VL-8B with a critic prompt): sees the images, the expert
evidence and both readers' answers; the rules are exposed to it as the ``run_consistency_rules`` tool
(PasteurAIze tool pattern). Its reopen list is merged with the rules' (union). If it fails, the
rules-only verdict stands. ``use_llm_critic=False`` is the "rules-only critic" ablation (no GPU pass).

The reopen is granted at most ``Settings.max_reopen`` times per image (MAST: no unbounded loops);
``state.reopen_count`` is the guard.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from pydantic_ai import Agent, ModelRetry, PromptedOutput, RunContext, capture_run_messages
from pydantic_ai.exceptions import UnexpectedModelBehavior
from pydantic_ai.models import Model
from pydantic_ai.settings import ModelSettings

from inspector.agents.llm import LLMAgent, response_texts
from inspector.config.settings import Settings
from inspector.schemas.agent_outputs import CriticOutput, ReaderOutput
from inspector.schemas.mmad import Question, SubTask
from inspector.schemas.state import InspectionState

_CELLS = [
    "top left", "top center", "top right", "middle left", "center",
    "middle right", "bottom left", "bottom center", "bottom right",
]

CRITIC_PROMPT = (
    "You are the quality-control reviewer of an industrial inspection team. Image 1 is the inspected "
    "part, image 2 a normal reference, image 3 (if present) the region a visual expert found most "
    "different. Two inspectors answered multiple-choice questions about image 1. First call "
    "`run_consistency_rules` to get the automatic checks. Then decide which questions must be "
    "re-asked because an answer contradicts the images, the expert evidence, or another answer of "
    "the same inspector. Only list question ids that appear in the input. Do not re-ask a question "
    "just because the two inspectors disagree."
)


def says_no_defect(q: Question, letter: str) -> bool:
    return q.options.get(letter, "").strip().lower().rstrip(".") in {"no", "no defect"}


def says_yes_defect(q: Question, letter: str) -> bool:
    return q.options.get(letter, "").strip().lower().rstrip(".") in {"yes", "defect"}


def cell_in_text(text: str) -> str | None:
    """Return the 3x3 cell named in ``text`` ("top-right part" -> "top right"), if any."""
    t = text.lower().replace("-", " ").replace("_", " ")
    for c in sorted(_CELLS, key=len, reverse=True):  # longest first so "top center" wins over "center"
        if c in t:
            return c
    return None


@dataclass
class CriticDeps:
    findings: list[str]        # rule findings, returned by the tool
    question_ids: set[str]     # valid ids for the output validator


class CriticVerifier(LLMAgent):
    name = "critic"

    def __init__(self, settings: Settings, model: Model | None = None):
        super().__init__(settings, settings.critic_model, model)

    # ---- LLM critic ------------------------------------------------------------
    def enabled(self) -> bool:
        return self.s.use_llm_critic

    def build(self, model: Model) -> Agent[CriticDeps, CriticOutput]:
        agent = Agent(
            model,
            name="Critic",
            system_prompt=CRITIC_PROMPT,
            output_type=PromptedOutput([CriticOutput]),
            deps_type=CriticDeps,
            retries={"output": self.s.json_retries},
            model_settings=ModelSettings(temperature=0.0, max_tokens=self.s.max_new_tokens),
        )

        @agent.tool
        def run_consistency_rules(ctx: RunContext[CriticDeps]) -> str:
            """Deterministic checks: missing answers, answers contradicting the visual expert."""
            return "\n".join(ctx.deps.findings) or "No rule violation found."

        @agent.output_validator
        def ids_exist(ctx: RunContext[CriticDeps], out: CriticOutput) -> CriticOutput:
            unknown = set(out.reopen_questions) - ctx.deps.question_ids
            if unknown:
                raise ModelRetry(f"Unknown question ids {sorted(unknown)}. Valid ids: {sorted(ctx.deps.question_ids)}.")
            return out

        return agent

    @staticmethod
    def describe(state: InspectionState) -> str:
        """Text input of the LLM critic: expert evidence + both readers' answers per question."""
        lines = [state.visual_evidence.as_text()] if state.visual_evidence else []
        for q in state.questions:
            opts = "; ".join(f"{k}. {v}" for k, v in sorted(q.options.items()))
            lines.append(f"[{q.id}] {q.text} ({opts})")
            for who, store in (("inspector A", state.reader_a_responses), ("inspector B", state.reader_b_responses)):
                a = store.get(q.id)
                lines.append(f"  {who}: " + (f"{a.selected_option} (conf {a.confidence:.2f}) {a.brief_rationale}" if a else "no valid answer"))
        return "\n".join(lines)

    # ---- rules -------------------------------------------------------------------
    def check_reader(self, state: InspectionState, answers: dict[str, ReaderOutput | None], who: str) -> tuple[bool, list[str], list[str]]:
        """Return (valid, reopen_ids, reasons) for one reader."""
        valid, reopen, reasons = True, [], []
        ev = state.visual_evidence
        strong = ev is not None and ev.confidence_score >= self.s.expert_flag_threshold
        has_loc = any(q.subtask == SubTask.DEFECT_LOCALIZATION for q in state.questions)
        for q in state.questions:
            a = answers.get(q.id)
            if a is None:
                valid = False
                reopen.append(q.id)
                reasons.append(f"{who}: no valid answer for {q.id}")
                continue
            if q.subtask == SubTask.ANOMALY_DISCRIMINATION and says_no_defect(q, a.selected_option):
                if strong:
                    reopen.append(q.id)
                    reasons.append(f"{who}: said no defect but the visual expert found a strong difference in the {ev.candidate_region}.")
                elif self.s.critic_use_question_structure and has_loc:
                    reopen.append(q.id)
                    reasons.append(f"{who}: said no defect but a defect-localization question exists.")
            if q.subtask == SubTask.DEFECT_LOCALIZATION and strong:
                cell = cell_in_text(q.options[a.selected_option])
                if cell and cell != ev.candidate_region:
                    reopen.append(q.id)
                    reasons.append(f"{who}: chose '{cell}' but the expert's strongest region is '{ev.candidate_region}'.")
        return valid, reopen, reasons

    def _run(self, state: InspectionState, rec: dict[str, Any]) -> InspectionState:
        valid, reopen, reasons = True, [], []
        for who, ans in (("reader_a", state.reader_a_responses), ("reader_b", state.reader_b_responses)):
            v, r, why = self.check_reader(state, ans, who)
            valid &= v
            reopen += r
            reasons += why

        if self.agent is not None:
            deps = CriticDeps(findings=list(reasons), question_ids={q.id for q in state.questions})
            prompt = self.describe(state)
            rec["prompt"] = prompt
            with capture_run_messages() as messages:
                try:
                    res = self.run_async(self.agent.run([prompt, *self.image_parts(state)], deps=deps))
                    usage = res.usage
                    rec.update(llm=res.output.model_dump(), tokens_in=usage.input_tokens, tokens_out=usage.output_tokens)
                    reopen += res.output.reopen_questions
                    if res.output.reopen_questions and res.output.critique_reason:
                        reasons.append(f"reviewer: {res.output.critique_reason}")
                except UnexpectedModelBehavior as e:  # LLM critic failed: the rules-only verdict stands
                    rec["llm_error"] = repr(e)
            rec["raw"] = response_texts(messages)

        reopen = sorted(set(reopen))
        allowed = state.reopen_count < self.s.max_reopen
        if reopen and allowed:
            state.reopen_count += 1  # the single reopen is granted here
        verdict = CriticOutput(
            is_valid=valid,
            conflict_detected=bool(reopen),
            reopen_questions=reopen if allowed else [],
            critique_reason=" ".join(reasons),
        )
        state.critic_verdict = verdict
        rec["verdict"] = verdict.model_dump()
        return state

In [ ]:
%%writefile inspector/agents/llm.py
"""Shared LLM plumbing for the pydantic-ai agents (same pattern as PasteurAIze).

* ``make_model``: an ``OpenAIChatModel`` pointed at an OpenAI-compatible endpoint (vLLM, Ollama...).
* ``image_part``: an image file as ``BinaryContent`` for a multimodal ``agent.run([...])``.
* ``VLLMServer``: one ``vllm serve <model>`` per GPU pass, so only one model is ever on the GPU.
* ``LLMAgent``: base for the reader and critic agents: builds the pydantic-ai ``Agent`` on
  ``load()``, owns one event loop for the whole pass, and stops the server on ``unload()``.
"""
from __future__ import annotations

import asyncio
import json
import mimetypes
import subprocess
import time
import urllib.request
from abc import abstractmethod
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

from pydantic_ai import Agent, BinaryContent
from pydantic_ai.messages import ModelMessage, ModelResponse, TextPart
from pydantic_ai.models import Model
from pydantic_ai.models.fallback import FallbackModel
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from inspector.agents.base import BaseAgent
from inspector.config.settings import Settings
from inspector.schemas.state import InspectionState


def make_model(s: Settings, model_name: str, ep: dict[str, str] | None = None) -> OpenAIChatModel:
    ep = ep or {}
    return OpenAIChatModel(
        ep.get("model", model_name),
        provider=OpenAIProvider(base_url=ep.get("base_url", s.llm_base_url), api_key=ep.get("api_key", s.llm_api_key)),
    )


def make_role_model(s: Settings, model_name: str, ep: dict[str, str] | None = None) -> Model:
    """``make_model``, plus a backup: a hosted ``remote`` endpoint (e.g. Groq) falls back to the default
    ``llm_*`` endpoint (e.g. Gemini) on any API error (429/402 quota, 5xx...)."""
    primary = make_model(s, model_name, ep)
    if ep and s.backup_enabled and s.llm_provider == "router" and s.llm_api_key not in ("", "EMPTY"):
        return FallbackModel(primary, make_model(s, model_name))
    return primary


def image_part(path: str) -> BinaryContent:
    media = mimetypes.guess_type(path)[0] or "image/png"
    return BinaryContent(data=Path(path).read_bytes(), media_type=media)


def response_texts(messages: list[ModelMessage]) -> list[str]:
    """Raw text of every model response (for the JSONL log; images are never logged)."""
    return [p.content for m in messages if isinstance(m, ModelResponse) for p in m.parts if isinstance(p, TextPart)]


def served_models(s: Settings) -> list[str] | None:
    """Model ids served at ``llm_base_url``, or None if nothing answers."""
    req = urllib.request.Request(s.llm_base_url.rstrip("/") + "/models", headers={"Authorization": f"Bearer {s.llm_api_key}"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return [m["id"] for m in json.load(r).get("data", [])]
    except OSError:
        return None


class VLLMServer:
    """Starts (if ``vllm_autostart``) or checks the endpoint for one model; ``stop()`` frees the GPU."""

    def __init__(self, s: Settings, model_name: str):
        self.s, self.model_name = s, model_name
        self.proc: subprocess.Popen | None = None

    def start(self) -> None:
        if not self.s.vllm_autostart:
            models = served_models(self.s)
            if models is None:
                raise RuntimeError(f"No LLM server at {self.s.llm_base_url}. Start `vllm serve {self.model_name}` or set ONEREF_VLLM_AUTOSTART=true.")
            if self.model_name not in models:
                raise RuntimeError(f"Server at {self.s.llm_base_url} serves {models}, not {self.model_name!r}.")
            return
        port = str(urlparse(self.s.llm_base_url).port or 8000)
        self.proc = subprocess.Popen(["vllm", "serve", self.model_name, "--port", port, *self.s.vllm_args])
        deadline = time.monotonic() + self.s.vllm_startup_timeout_s
        while time.monotonic() < deadline:
            if self.proc.poll() is not None:
                raise RuntimeError(f"vllm serve {self.model_name} exited with code {self.proc.returncode}")
            if self.model_name in (served_models(self.s) or []):
                return
            time.sleep(5)
        self.stop()
        raise TimeoutError(f"vllm serve {self.model_name} not ready after {self.s.vllm_startup_timeout_s}s")

    def stop(self) -> None:
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        self.proc = None


class LLMAgent(BaseAgent):
    """An agent backed by a pydantic-ai ``Agent``. ``model`` injection = no server (mock / tests)."""

    def __init__(self, settings: Settings, model_name: str, model: Model | None = None):
        self.s = settings
        self.model_name = model_name
        self._model = model
        self.agent: Agent | None = None
        self.server: VLLMServer | None = None
        self.loop: asyncio.AbstractEventLoop | None = None

    @abstractmethod
    def build(self, model: Model) -> Agent:
        """Create the pydantic-ai Agent (prompt, output_type, tools, validators)."""

    def enabled(self) -> bool:
        return True

    def load(self) -> None:
        if not self.enabled():
            return
        model = self._model
        if model is None:
            ep = self.s.remote.get(self.name.removesuffix("_reopen"))  # hosted API: no local server
            if ep is None and self.s.llm_provider == "vllm":
                self.server = VLLMServer(self.s, self.model_name)
                self.server.start()
            model = make_role_model(self.s, self.model_name, ep)
        self.agent = self.build(model)
        self.loop = asyncio.new_event_loop()  # one loop per pass: the HTTP client stays bound to it

    def unload(self) -> None:
        if self.server:
            self.server.stop()
        if self.loop:
            self.loop.close()
        self.agent = self.server = self.loop = None
        super().unload()

    def image_parts(self, state: InspectionState) -> list[BinaryContent]:
        """Query, normal reference (random_templates[0]) and the expert's zoomed crop."""
        ev = state.visual_evidence
        paths = [state.query_image_path, state.reference_image_path] + ([ev.crop_path] if ev else [])
        return [image_part(p) for p in paths]

    def run_async(self, coro: Any) -> Any:
        assert self.loop is not None, "load() must be called before run()"
        return self.loop.run_until_complete(coro)

In [ ]:
%%writefile inspector/agents/mock.py
"""Deterministic GPU-free stand-ins for smoke runs and tests. Never read ground truth.

Readers and critic are the REAL pydantic-ai agents (prompt, output schema, validators, tool),
driven by a ``FunctionModel`` instead of a served MLLM.
"""
from __future__ import annotations

import hashlib
import json
import re
from typing import Any

from pydantic_ai.messages import ModelMessage, ModelRequest, ModelResponse, TextPart, ToolCallPart, ToolReturnPart, UserPromptPart
from pydantic_ai.models.function import AgentInfo, FunctionModel

from inspector.agents.base import BaseAgent
from inspector.agents.critic_verifier import CriticVerifier
from inspector.agents.reader_mllm import ReaderMLLM
from inspector.config.settings import Settings
from inspector.schemas.agent_outputs import VisualExpertOutput
from inspector.schemas.state import InspectionState

_OPTION_RE = re.compile(r"^([A-D])\. ", re.MULTILINE)


def _h(*parts: str) -> int:
    return int(hashlib.md5("|".join(parts).encode()).hexdigest(), 16)


def _user_text(messages: list[ModelMessage]) -> str:
    """Text of the first user prompt (the images are BinaryContent items, skipped)."""
    for m in messages:
        if isinstance(m, ModelRequest):
            for p in m.parts:
                if isinstance(p, UserPromptPart):
                    return p.content if isinstance(p.content, str) else " ".join(c for c in p.content if isinstance(c, str))
    return ""


def reader_model(seed: str) -> FunctionModel:
    """Answers a pseudo-random option that the question offers."""

    def fn(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
        text = _user_text(messages)
        letters = sorted(set(_OPTION_RE.findall(text))) or ["A"]
        letter = letters[_h(seed, text) % len(letters)]
        return ModelResponse(parts=[TextPart(json.dumps({"selected_option": letter, "confidence": 0.5, "brief_rationale": "mock"}))])

    return FunctionModel(fn)


def critic_model() -> FunctionModel:
    """Calls ``run_consistency_rules`` first, then approves (no extra reopen)."""

    def fn(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
        called = any(isinstance(p, ToolReturnPart) for m in messages if isinstance(m, ModelRequest) for p in m.parts)
        if not called:
            return ModelResponse(parts=[ToolCallPart(tool_name="run_consistency_rules", args={})])
        out = {"is_valid": True, "conflict_detected": False, "reopen_questions": [], "critique_reason": "mock review"}
        return ModelResponse(parts=[TextPart(json.dumps(out))])

    return FunctionModel(fn)


class MockExpert(BaseAgent):
    name = "visual_expert"

    def _run(self, state: InspectionState, rec: dict[str, Any]) -> InspectionState:
        h = _h(state.image_id)
        cells = ["top left", "top center", "top right", "middle left", "center", "middle right", "bottom left", "bottom center", "bottom right"]
        state.visual_evidence = VisualExpertOutput(
            heatmap_path="mock_heatmap.png", candidate_region=cells[h % 9],  # type: ignore[arg-type]
            confidence_score=(h % 100) / 100, crop_path="mock_crop.png",
        )
        return state


class MockReader(ReaderMLLM):
    def __init__(self, settings: Settings, name: str, reopen: bool = False):
        super().__init__(settings, model_name="mock", name=name, reopen=reopen, model=reader_model(name))

    def image_parts(self, state: InspectionState) -> list:
        return []  # mock runs have no image files


class MockCritic(CriticVerifier):
    def __init__(self, settings: Settings):
        super().__init__(settings, model=critic_model())

    def image_parts(self, state: InspectionState) -> list:
        return []

In [ ]:
%%writefile inspector/agents/reader_mllm.py
"""MLLM reader agent (Reader A = Qwen3-VL-8B, Reader B = InternVL3-8B), pydantic-ai style.

One class, two instances. Each question is one ``agent.run([prompt, query, reference, crop])``:
the model answers in JSON (``PromptedOutput``), pydantic-ai validates it against ``ReaderAnswer``
and an ``output_validator`` rejects options the question does not offer (``ModelRetry``).
After ``json_retries`` failed retries the answer is ``None`` (counts as an error downstream).

``reopen=True`` makes this a *reopen pass*: only the questions flagged by the critic are re-asked,
with the critic's reason added to the prompt.
"""
from __future__ import annotations

import asyncio
from typing import Any

from pydantic_ai import Agent, ModelRetry, PromptedOutput, RunContext, capture_run_messages
from pydantic_ai.exceptions import UnexpectedModelBehavior
from pydantic_ai.models import Model
from pydantic_ai.settings import ModelSettings

from inspector.agents.llm import LLMAgent, response_texts
from inspector.config.settings import Settings
from inspector.schemas.agent_outputs import ReaderAnswer, ReaderOutput
from inspector.schemas.mmad import Question
from inspector.schemas.state import InspectionState

SYSTEM_PROMPT = (
    "You are an industrial quality inspector. Image 1 is the part to inspect. Image 2 is a normal "
    "(defect-free) reference of the same product. Image 3, if present, is a zoomed crop of the region "
    "that differs most from the reference. Answer the multiple-choice question with the letter of one "
    "of the given options, a confidence between 0 and 1, and one short sentence of rationale."
)


def build_prompt(q: Question, evidence_text: str | None, critique: str | None = None) -> str:
    opts = "\n".join(f"{k}. {v}" for k, v in sorted(q.options.items()))
    parts = []
    if evidence_text:
        parts.append(evidence_text)
    if critique:
        parts.append(f"A reviewer flagged a possible inconsistency in your previous answers: {critique} Reconsider carefully.")
    parts.append(f"Question: {q.text}\nOptions:\n{opts}")
    return "\n\n".join(parts)


class ReaderMLLM(LLMAgent):
    def __init__(self, settings: Settings, model_name: str, name: str, reopen: bool = False, model: Model | None = None):
        super().__init__(settings, model_name, model)
        self.name = name
        self.reopen = reopen

    def build(self, model: Model) -> Agent[Question, ReaderAnswer]:
        agent = Agent(
            model,
            name=self.name,
            system_prompt=SYSTEM_PROMPT,
            output_type=PromptedOutput([ReaderAnswer]),
            deps_type=Question,
            retries={"output": self.s.json_retries},
            model_settings=ModelSettings(temperature=0.0, max_tokens=self.s.max_new_tokens),
        )

        @agent.output_validator
        def option_is_offered(ctx: RunContext[Question], out: ReaderAnswer) -> ReaderAnswer:
            if out.selected_option not in ctx.deps.options:
                raise ModelRetry(f"'{out.selected_option}' is not an option. Choose one of {sorted(ctx.deps.options)}.")
            return out

        return agent

    # ---- which dict of the state this reader owns ---------------------------
    def _store(self, state: InspectionState) -> dict[str, ReaderOutput | None]:
        return state.reader_a_responses if "reader_a" in self.name else state.reader_b_responses

    def applies(self, state: InspectionState) -> bool:
        if not self.reopen:
            return True
        return bool(state.critic_verdict and state.critic_verdict.reopen_questions)

    # ---- agent ---------------------------------------------------------------
    async def ask(self, q: Question, state: InspectionState, critique: str | None, images: list, rec: dict[str, Any]) -> ReaderOutput | None:
        ev = state.visual_evidence
        prompt = build_prompt(q, ev.as_text() if ev else None, critique)
        call: dict[str, Any] = {"question_id": q.id, "prompt": prompt}
        rec.setdefault("calls", []).append(call)
        with capture_run_messages() as messages:
            try:
                result = await self.agent.run([prompt, *images], deps=q)
            except UnexpectedModelBehavior as e:  # invalid after retries => counted as an error
                call.update(raw=response_texts(messages), error=repr(e))
                return None
        usage = result.usage
        call.update(raw=response_texts(messages), tokens_in=usage.input_tokens, tokens_out=usage.output_tokens)
        rec["tokens_in"] = rec.get("tokens_in", 0) + usage.input_tokens
        rec["tokens_out"] = rec.get("tokens_out", 0) + usage.output_tokens
        return ReaderOutput(question_id=q.id, **result.output.model_dump())

    def _run(self, state: InspectionState, rec: dict[str, Any]) -> InspectionState:
        store = self._store(state)
        if self.reopen:
            ids = set(state.critic_verdict.reopen_questions) if state.critic_verdict else set()
            critique = state.critic_verdict.critique_reason if state.critic_verdict else None
            todo = [q for q in state.questions if q.id in ids]
        else:
            critique, todo = None, state.questions
        images = self.image_parts(state)  # read once per image, shared by its questions

        async def ask_all() -> list[ReaderOutput | None]:  # concurrent => vLLM batches the questions
            return await asyncio.gather(*(self.ask(q, state, critique, images, rec) for q in todo))

        for q, out in zip(todo, self.run_async(ask_all())):
            store[q.id] = out
        rec["answers"] = {q.id: (store[q.id].selected_option if store.get(q.id) else None) for q in todo}
        return state

In [ ]:
%%writefile inspector/agents/visual_expert.py
"""Visual expert agent: training-free patch comparison with a frozen DINOv3 (DINOv2 fallback).

Algorithm
---------
1. Resize query ``I`` and reference ``R`` to a multiple of the patch size and extract per-patch
   features (L2-normalized).
2. For every query patch, anomaly = ``1 - max_j cos(f_I[p], f_R[j])`` (distance to the closest
   reference patch, so pose shifts are tolerated).
3. Upsample to the image size -> heatmap; the argmax gives the 3x3 grid cell and a zoomed crop.
4. ``confidence_score`` = peak distance linearly rescaled to [0, 1] with calibration constants
   from ``Settings`` (tune on the dev split only).
"""
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

from inspector.agents.base import BaseAgent
from inspector.config.settings import Settings
from inspector.schemas.agent_outputs import GridCell, VisualExpertOutput
from inspector.schemas.state import InspectionState

_GRID: list[list[GridCell]] = [
    ["top left", "top center", "top right"],
    ["middle left", "center", "middle right"],
    ["bottom left", "bottom center", "bottom right"],
]
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def grid_cell(y: int, x: int, h: int, w: int) -> GridCell:
    """Map a pixel (y, x) of an h x w image to its 3x3 cell name."""
    return _GRID[min(2, y * 3 // h)][min(2, x * 3 // w)]


def anomaly_map(q: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Per-patch min cosine distance. ``q`` (Nq, D), ``r`` (Nr, D) -> (Nq,) in [0, 2]."""
    q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-8)
    r = r / (np.linalg.norm(r, axis=1, keepdims=True) + 1e-8)
    return 1.0 - (q @ r.T).max(axis=1)


class VisualExpert(BaseAgent):
    name = "visual_expert"

    def __init__(self, settings: Settings, out_dir: Path):
        self.s = settings
        self.out_dir = Path(out_dir) / "expert_artifacts"
        self.model: Any = None
        self.patch = 16
        self.n_prefix = 1  # CLS (+ register tokens for DINOv3)
        self.size = 448    # input side in pixels (multiple of 14 and 16 after rounding)
        self.torch: Any = None
        self.device = "cpu"

    # ---- lifecycle -----------------------------------------------------------
    def load(self) -> None:
        import torch
        from transformers import AutoModel

        self.torch = torch
        self.device = self.s.device if (self.s.device != "cuda" or torch.cuda.is_available()) else "cpu"
        dtype = getattr(torch, self.s.dtype) if self.device == "cuda" else torch.float32
        last_err: Exception | None = None
        for model_id in (self.s.expert_model, self.s.expert_fallback_model):
            try:  # DINOv3 is gated: if the license was not accepted we fall back to DINOv2
                self.model = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(self.device).eval()
                self.model_id = model_id
                break
            except Exception as e:  # noqa: BLE001
                last_err = e
        if self.model is None:
            raise RuntimeError(f"Could not load expert model: {last_err!r}")
        cfg = self.model.config
        ps = cfg.patch_size
        self.patch = ps if isinstance(ps, int) else ps[0]
        self.n_prefix = 1 + int(getattr(cfg, "num_register_tokens", 0))
        self.size = (448 // self.patch) * self.patch
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def unload(self) -> None:
        self.model = None
        super().unload()

    # ---- feature extraction --------------------------------------------------
    def _features(self, img: Image.Image) -> np.ndarray:
        arr = (np.asarray(img.convert("RGB").resize((self.size, self.size), Image.BICUBIC), dtype=np.float32) / 255.0 - _MEAN) / _STD
        x = self.torch.from_numpy(arr).permute(2, 0, 1)[None].to(self.device, self.model.dtype)
        with self.torch.no_grad():
            h = self.model(pixel_values=x).last_hidden_state[0]
        return h[self.n_prefix :].float().cpu().numpy()  # (grid*grid, D)

    # ---- agent ---------------------------------------------------------------
    def _run(self, state: InspectionState, rec: dict[str, Any]) -> InspectionState:
        query = Image.open(state.query_image_path).convert("RGB")
        ref = Image.open(state.reference_image_path).convert("RGB")
        out = self.compute(query, ref, state.image_id)
        rec["model"] = getattr(self, "model_id", "?")
        rec["output"] = out.model_dump()
        state.visual_evidence = out
        return state

    def compute(self, query: Image.Image, ref: Image.Image, image_id: str) -> VisualExpertOutput:
        """Pure image-in / evidence-out step (also usable from a notebook)."""
        dist = anomaly_map(self._features(query), self._features(ref))
        g = self.size // self.patch
        amap = dist.reshape(g, g)
        w, h = query.size
        up = np.asarray(Image.fromarray(amap.astype(np.float32), mode="F").resize((w, h), Image.BICUBIC))
        py, px = np.unravel_index(int(up.argmax()), up.shape)
        peak = float(amap.max())
        score = float(np.clip((peak - self.s.anomaly_norm_low) / (self.s.anomaly_norm_high - self.s.anomaly_norm_low), 0, 1))

        stem = image_id.replace("/", "__")
        heat_path = self.out_dir / f"{stem}_heatmap.png"
        crop_path = self.out_dir / f"{stem}_crop.png"
        norm = np.clip((up - up.min()) / (np.ptp(up) + 1e-8), 0, 1)
        overlay = np.asarray(query, dtype=np.float32).copy()
        overlay[..., 0] = np.clip(overlay[..., 0] * (1 - 0.6 * norm) + 255 * 0.6 * norm, 0, 255)
        Image.fromarray(overlay.astype(np.uint8)).save(heat_path)

        cw, ch = max(w // 3, 32), max(h // 3, 32)
        x0 = int(np.clip(px - cw // 2, 0, max(w - cw, 0)))
        y0 = int(np.clip(py - ch // 2, 0, max(h - ch, 0)))
        query.crop((x0, y0, x0 + cw, y0 + ch)).resize((cw * 2, ch * 2), Image.BICUBIC).save(crop_path)

        return VisualExpertOutput(
            heatmap_path=str(heat_path),
            candidate_region=grid_cell(int(py), int(px), h, w),
            confidence_score=score,
            crop_path=str(crop_path),
        )

In [ ]:
%%writefile inspector/config/settings.py
"""Global configuration (Pydantic Settings).

Every value can be overridden by an env var prefixed ``ONEREF_`` (e.g.
``ONEREF_DEVICE=cpu``) or by a ``.env`` file.
"""
from __future__ import annotations

from pathlib import Path
from typing import Literal

from pydantic import AliasChoices, Field, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict

ROOT = Path(__file__).resolve().parents[3]  # OneRef/


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="ONEREF_", env_file=".env", extra="ignore", populate_by_name=True)

    # ---- paths -------------------------------------------------------------
    mmad_root: Path = ROOT / "data" / "MMAD"          # images (gitignored)
    mmad_json: Path = ROOT / "data" / "MMAD" / "mmad.json"
    splits_dir: Path = ROOT / "data" / "splits"
    outputs_dir: Path = ROOT / "outputs"              # per-agent cache + JSONL (gitignored)

    # ---- hardware / pass budget -------------------------------------------
    device: str = "cuda"                              # falls back to cpu if CUDA is unavailable
    dtype: Literal["bfloat16", "float16", "float32"] = "bfloat16"   # visual expert only
    max_models_on_gpu: int = 1                        # pass-based execution: one model at a time

    # ---- models ------------------------------------------------------------
    expert_model: str = "facebook/dinov3-vitb16-pretrain-lvd1689m"
    expert_fallback_model: str = "facebook/dinov2-base"
    reader_a_model: str = "Qwen/Qwen3-VL-8B-Instruct"
    reader_b_model: str = "OpenGVLab/InternVL3-8B"
    critic_model: str = "Qwen/Qwen3-VL-8B-Instruct"  # same weights as reader A, critic prompt
    max_new_tokens: int = 256
    json_retries: int = 1                             # pydantic-ai output retries on bad JSON / option

    # ---- LLM endpoint (OpenAI-compatible) ----------------------------------
    # "router": Hugging Face router (https://router.huggingface.co/v1), key = HF_TOKEN, no local server.
    #           Model ids are HF ids, optionally with a provider suffix ("Qwen/Qwen3-VL-8B-Instruct:novita").
    # "vllm":   a local/Colab vLLM server (see vllm_* below); set ONEREF_LLM_BASE_URL=http://localhost:8000/v1
    #           and ONEREF_LLM_API_KEY=EMPTY.
    llm_provider: Literal["router", "vllm"] = "router"
    llm_base_url: str = "https://router.huggingface.co/v1"
    llm_api_key: str = Field("EMPTY", validation_alias=AliasChoices("ONEREF_LLM_API_KEY", "HF_TOKEN"))
    # True: each GPU pass starts `vllm serve <model>` and stops it at the end of the pass.
    # False: a server is already running; the pass only checks it serves the right model.
    vllm_autostart: bool = False
    vllm_args: list[str] = [
        "--max-model-len", "8192", "--gpu-memory-utilization", "0.9", "--trust-remote-code",
        "--enable-auto-tool-choice", "--tool-call-parser", "hermes",  # needed by the critic's tool
    ]
    vllm_startup_timeout_s: int = 900
    # Hosted API per agent, e.g. ONEREF_REMOTE='{"reader_a":{"base_url":"https://api.x.ai/v1","api_key":"...","model":"grok-..."}}'
    # Roles: reader_a, reader_b, critic. A listed role skips vLLM entirely (nothing on the GPU).
    remote: dict[str, dict[str, str]] = {}

    # A role with a `remote` endpoint falls back to the default llm_* endpoint on API errors (quota...).
    backup_enabled: bool = True

    # ---- Grok (xAI, OpenAI-compatible) -------------------------------------
    # ONEREF_USE_GROK=true fills `remote` for the three roles (explicit ONEREF_REMOTE entries win).
    # Hosted: leaves the strict local <=8B protocol, report as upper bound. Model ids are placeholders.
    use_grok: bool = False
    grok_base_url: str = "https://api.x.ai/v1"
    grok_api_key: str = Field("", validation_alias=AliasChoices("ONEREF_GROK_API_KEY", "XAI_API_KEY"))
    grok_reader_a_model: str = "grok-4"
    grok_reader_b_model: str = "grok-4"
    grok_critic_model: str = "grok-4"

    # ---- expert calibration (tune on dev only) ----------------------------
    anomaly_norm_low: float = 0.2                     # cosine distance mapped to score 0
    anomaly_norm_high: float = 0.8                    # cosine distance mapped to score 1
    expert_flag_threshold: float = 0.5                # score >= threshold => "defect" vote

    # ---- protocol ----------------------------------------------------------
    dev_fraction: float = Field(0.10, gt=0, lt=1)
    seed: int = 0
    max_reopen: int = 1                               # MAST: no unbounded loops

    # ---- aggregation weights: {agent: {subtask: weight}}, default 1.0 -----
    default_weight: float = 1.0
    weights: dict[str, dict[str, float]] = Field(default_factory=dict)  # fixed on dev, e.g. {"reader_a": {"defect_localization": 1.3}}
    expert_weight_discrimination: float = 1.0

    # ---- critic ------------------------------------------------------------
    # Rule "discrimination=No but the image has defect-localization questions" depends on the
    # *presence* of questions, which reveals the label in MMAD. Off by default (strict protocol).
    critic_use_question_structure: bool = False
    use_llm_critic: bool = True                       # False = rules-only critic (ablation, no GPU pass)

    # ---- telemetry ---------------------------------------------------------
    use_logfire: bool = False                         # optional; JSONL is always on

    @model_validator(mode="after")
    def _grok_remote(self) -> "Settings":
        if not self.use_grok:
            return self
        if not self.grok_api_key:
            raise ValueError("ONEREF_USE_GROK=true needs ONEREF_GROK_API_KEY (or XAI_API_KEY)")
        for role, model in (("reader_a", self.grok_reader_a_model), ("reader_b", self.grok_reader_b_model), ("critic", self.grok_critic_model)):
            self.remote.setdefault(role, {"base_url": self.grok_base_url, "api_key": self.grok_api_key, "model": model})
        return self


def get_settings() -> Settings:
    return Settings()

In [ ]:
%%writefile inspector/main.py
"""CLI entry point.

    python -m inspector.main --split dev  --limit 50 --run-name dev01
    python -m inspector.main --split test --frozen-config --run-name final     # run ONCE
    python -m inspector.main --split dev --limit 5 --mock                       # GPU-free smoke test

Needs ``src`` on PYTHONPATH (``pip install -e .`` or ``set PYTHONPATH=src``).
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

from inspector.config.settings import get_settings
from inspector.orchestrator.graph import build_steps
from inspector.orchestrator.pass_runner import PassRunner
from inspector.schemas.mmad import load_mmad
from inspector.schemas.state import InspectionState
from inspector.utils import metrics
from inspector.utils.splits import make_split, save_split
from inspector.utils.telemetry import Telemetry


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="OneRef multi-agent inspector (strict MMAD 1-shot)")
    ap.add_argument("--split", choices=["dev", "test"], default="dev")
    ap.add_argument("--limit", type=int, default=None, help="use only the first N images of the split")
    ap.add_argument("--available-only", action="store_true", help="keep only images present under ONEREF_MMAD_ROOT (sample download)")
    ap.add_argument("--run-name", default="run")
    ap.add_argument("--mock", action="store_true", help="GPU-free mock agents (smoke test)")
    ap.add_argument("--frozen-config", action="store_true", help="required for --split test: config is frozen")
    args = ap.parse_args(argv)

    if args.split == "test" and not args.frozen_config:
        print("Refusing to run the test split without --frozen-config (tune on dev only).", file=sys.stderr)
        return 2

    s = get_settings()
    samples = load_mmad(s.mmad_json, available_under=s.mmad_root if args.available_only else None)
    if args.available_only:
        print(f"--available-only: {len(samples)} images found under {s.mmad_root}")
        if not samples:
            print("No image of mmad.json exists under ONEREF_MMAD_ROOT: download a sample first "
                  "(scripts/download_mmad.py --sample) and check the folder layout.", file=sys.stderr)
            return 2
    dev, test = make_split(samples, s.dev_fraction, s.seed)
    save_split(dev, test, s.splits_dir)
    chosen = (dev if args.split == "dev" else test)[: args.limit]

    run_dir = Path(s.outputs_dir) / args.run_name
    tel = Telemetry(run_dir / "telemetry.jsonl", run=args.run_name, use_logfire=s.use_logfire)
    states = [InspectionState.from_sample(x, str(s.mmad_root)) for x in chosen]
    states = PassRunner(run_dir, tel).run(build_steps(s, run_dir, mock=args.mock), states)

    rows = metrics.build_rows(chosen, states)
    acc = metrics.per_subtask_accuracy(rows)
    for k, v in acc.items():
        print(f"{k.value:26s} {v:6.1%}")
    lo, hi = metrics.bootstrap_ci(rows)
    print(f"{'MEAN (7 subtasks)':26s} {metrics.mean_accuracy(rows):6.1%}  95% CI [{lo:.1%}, {hi:.1%}]  n_img={len(states)}")
    tel.write({"event": "final_metrics", "mean": metrics.mean_accuracy(rows), "per_subtask": {k.value: v for k, v in acc.items()}})
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile inspector/orchestrator/graph.py
"""Deterministic workflow definition: an explicit, ordered list of typed steps (no free-form loops).

    expert -> reader_a -> reader_b -> critic -> [reader_a_reopen -> reader_b_reopen] -> aggregator

The only "loop" is the reopen pair, which is a straight-line pass that runs only for images the
critic flagged, and the critic grants it at most ``max_reopen`` (=1) times per image.
"""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable

from inspector.agents.aggregator import Aggregator
from inspector.agents.base import BaseAgent
from inspector.agents.critic_verifier import CriticVerifier
from inspector.config.settings import Settings


@dataclass(frozen=True)
class Step:
    name: str                       # also the cache sub-folder name
    make_agent: Callable[[], BaseAgent]  # built lazily: nothing heavy is created until the pass starts


def build_steps(settings: Settings, run_dir: Path, mock: bool = False) -> list[Step]:
    if mock:
        from inspector.agents.mock import MockCritic, MockExpert, MockReader

        expert: Callable[[], BaseAgent] = MockExpert
        reader = lambda name, reopen: (lambda: MockReader(settings, name, reopen))  # noqa: E731
        critic: Callable[[], BaseAgent] = lambda: MockCritic(settings)  # noqa: E731
    else:
        from inspector.agents.reader_mllm import ReaderMLLM
        from inspector.agents.visual_expert import VisualExpert

        expert = lambda: VisualExpert(settings, run_dir)  # noqa: E731
        models = {"reader_a": settings.reader_a_model, "reader_b": settings.reader_b_model}
        reader = lambda name, reopen: (lambda: ReaderMLLM(settings, models[name.removesuffix("_reopen")], name, reopen))  # noqa: E731
        critic = lambda: CriticVerifier(settings)  # noqa: E731

    return [
        Step("1_expert", expert),
        Step("2_reader_a", reader("reader_a", False)),
        Step("3_reader_b", reader("reader_b", False)),
        Step("4_critic", critic),
        Step("5_reader_a_reopen", reader("reader_a_reopen", True)),
        Step("6_reader_b_reopen", reader("reader_b_reopen", True)),
        Step("7_aggregator", lambda: Aggregator(settings)),
    ]

In [ ]:
%%writefile inspector/orchestrator/pass_runner.py
"""Pass-based runner: each step runs over the WHOLE split, then frees the GPU.

Peak VRAM = one model (an 8B MLLM, or DINOv3), never two. After every pass the full
``InspectionState`` of each image is cached to ``<run_dir>/<step>/<image_id>.json``, so a crashed or
disconnected Colab session resumes without recomputing finished images, and baselines can reuse
the readers' cached answers.
"""
from __future__ import annotations

from pathlib import Path

from inspector.agents.base import free_gpu
from inspector.orchestrator.graph import Step
from inspector.schemas.state import InspectionState
from inspector.utils.telemetry import Telemetry


class PassRunner:
    def __init__(self, run_dir: Path, tel: Telemetry):
        self.run_dir = Path(run_dir)
        self.tel = tel

    def _cache(self, step: Step, image_id: str) -> Path:
        return self.run_dir / step.name / (image_id.replace("/", "__") + ".json")

    def run(self, steps: list[Step], states: list[InspectionState]) -> list[InspectionState]:
        for step in steps:
            states = self._run_step(step, states)
        return states

    def _run_step(self, step: Step, states: list[InspectionState]) -> list[InspectionState]:
        done: dict[str, InspectionState] = {}
        pending: list[InspectionState] = []
        for s in states:
            p = self._cache(step, s.image_id)
            if p.exists():
                done[s.image_id] = InspectionState.model_validate_json(p.read_text(encoding="utf-8"))
            else:
                pending.append(s)

        agent = step.make_agent()
        todo = [s for s in pending if agent.applies(s)]
        self.tel.write({"event": "pass_start", "step": step.name, "cached": len(done), "to_run": len(todo)})
        if todo:  # skip loading weights entirely when nothing needs this model
            agent.load()
            try:
                for s in todo:
                    done[s.image_id] = agent.run(s, self.tel)
            finally:  # always release the GPU, even on error, before the next pass
                agent.unload()
        del agent
        free_gpu()

        for s in pending:
            res = done.setdefault(s.image_id, s)  # images the agent skipped pass through unchanged
            p = self._cache(step, s.image_id)
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(res.model_dump_json(), encoding="utf-8")
        self.tel.write({"event": "pass_end", "step": step.name})
        return [done[s.image_id] for s in states]

In [ ]:
%%writefile inspector/schemas/agent_outputs.py
"""Typed messages exchanged between agents. Every inter-agent payload is validated here."""
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, ConfigDict, Field

from inspector.schemas.mmad import OptionLetter

# 3x3 grid cell names, row-major.
GridCell = Literal[
    "top left", "top center", "top right",
    "middle left", "center", "middle right",
    "bottom left", "bottom center", "bottom right",
]


class VisualExpertOutput(BaseModel):
    """Evidence from the DINOv3 expert: where the query differs from the reference."""

    heatmap_path: str
    candidate_region: GridCell
    confidence_score: float = Field(ge=0.0, le=1.0)  # normalized anomaly peak
    crop_path: str                                   # zoomed crop around the peak
    crop_base64: str | None = None                   # optional inline form of the crop

    def as_text(self) -> str:
        """Text injected into the readers' prompt."""
        level = "high" if self.confidence_score >= 0.66 else "medium" if self.confidence_score >= 0.33 else "low"
        return (
            f"Visual expert (automatic patch comparison with the reference): "
            f"most different region = {self.candidate_region}, "
            f"difference level = {level} ({self.confidence_score:.2f}). "
            "This is a hint, not ground truth."
        )


class ReaderAnswer(BaseModel):
    """What the MLLM itself returns (the pydantic-ai ``output_type`` of a reader)."""

    model_config = ConfigDict(extra="ignore")

    selected_option: OptionLetter
    confidence: float = Field(ge=0.0, le=1.0)
    brief_rationale: str = ""


class ReaderOutput(ReaderAnswer):
    """One MLLM answer to one question, tagged with the question it answers."""

    question_id: str


class CriticOutput(BaseModel):
    is_valid: bool                       # schema/format checks passed
    conflict_detected: bool              # logical inconsistency found
    reopen_questions: list[str] = Field(default_factory=list)  # question ids
    critique_reason: str = ""


class FinalAnswer(BaseModel):
    question_id: str
    selected_option: OptionLetter | None  # None = no valid answer (counts as an error)
    votes: dict[str, float] = Field(default_factory=dict)  # option -> weight


class FinalReport(BaseModel):
    """Per-image inspection report."""

    image_id: str
    answers: list[FinalAnswer]
    expert: VisualExpertOutput | None = None
    critic: CriticOutput | None = None
    reopened: bool = False

In [ ]:
%%writefile inspector/schemas/mmad.py
"""Pydantic models for ``mmad.json`` (MMAD benchmark, strict 1-shot protocol).

Strict protocol: the only reference image ever exposed is ``random_templates[0]``.
``similar_templates`` is deliberately NOT modelled so it cannot leak into a prompt.
"""
from __future__ import annotations

import json
from enum import Enum
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, model_validator

OptionLetter = Literal["A", "B", "C", "D"]


class SubTask(str, Enum):
    """The 7 MMAD subtasks (Object Structure/Details are merged into object_analysis)."""

    ANOMALY_DISCRIMINATION = "anomaly_discrimination"
    DEFECT_CLASSIFICATION = "defect_classification"
    DEFECT_LOCALIZATION = "defect_localization"
    DEFECT_DESCRIPTION = "defect_description"
    DEFECT_ANALYSIS = "defect_analysis"
    OBJECT_CLASSIFICATION = "object_classification"
    OBJECT_ANALYSIS = "object_analysis"


# Raw MMAD "type" strings -> SubTask (matched case-insensitively, spaces/underscores ignored).
_RAW_TYPES: dict[str, SubTask] = {
    "anomalydiscrimination": SubTask.ANOMALY_DISCRIMINATION,
    "defectclassification": SubTask.DEFECT_CLASSIFICATION,
    "defectlocalization": SubTask.DEFECT_LOCALIZATION,
    "defectdescription": SubTask.DEFECT_DESCRIPTION,
    "defectanalysis": SubTask.DEFECT_ANALYSIS,
    "objectclassification": SubTask.OBJECT_CLASSIFICATION,
    "objectanalysis": SubTask.OBJECT_ANALYSIS,
    "objectstructure": SubTask.OBJECT_ANALYSIS,
    "objectdetails": SubTask.OBJECT_ANALYSIS,
}


def parse_subtask(raw: str) -> SubTask:
    key = raw.replace(" ", "").replace("_", "").lower()
    try:
        return _RAW_TYPES[key]
    except KeyError as e:  # unknown type must fail loudly, never be silently scored
        raise ValueError(f"Unknown MMAD subtask type: {raw!r}") from e


class Question(BaseModel):
    """One multiple-choice question. ``options`` maps letter -> text (2 or 4 entries)."""

    model_config = ConfigDict(frozen=True)

    id: str                                   # "<image_id>#<index>"
    text: str
    options: dict[OptionLetter, str] = Field(min_length=2, max_length=4)
    answer: OptionLetter | None = None        # ground truth, used ONLY by metrics
    subtask: SubTask

    @model_validator(mode="after")
    def _answer_in_options(self) -> "Question":
        if self.answer is not None and self.answer not in self.options:
            raise ValueError(f"answer {self.answer!r} not in options {list(self.options)}")
        return self


class Sample(BaseModel):
    """One MMAD image with its questions and its single normal reference."""

    image_id: str
    image_path: str
    mask_path: str | None = None              # evaluation only, never given to agents
    source: str = "unknown"                   # MVTec / VisA / MVTec LOCO / GoodsAD
    category: str = "unknown"
    is_defective: bool = False
    templates: list[str] = Field(min_length=1)  # random_templates, order preserved
    questions: list[Question]

    @property
    def reference(self) -> str:
        """Strict 1-shot: always ``random_templates[0]``."""
        return self.templates[0]


def _parse_options(raw: dict | list) -> dict[str, str]:
    """MMAD stores options as {"A": "...", ...}; tolerate a plain list too."""
    if isinstance(raw, dict):
        return {str(k).strip().upper()[:1]: str(v) for k, v in raw.items()}
    return {"ABCD"[i]: str(v) for i, v in enumerate(raw)}


def load_mmad(path: str | Path, limit: int | None = None, available_under: str | Path | None = None) -> list[Sample]:
    """Load ``mmad.json`` (dict keyed by image path) into validated ``Sample`` objects.

    ``available_under``: keep only images whose file and reference template exist under that folder
    (for a downloaded sample of MMAD).
    """
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    items = data.items() if isinstance(data, dict) else ((d["image"], d) for d in data)
    samples: list[Sample] = []
    for image_path, entry in items:
        if available_under is not None:
            refs = entry.get("random_templates") or [None]
            if not all((Path(available_under) / p).is_file() for p in (image_path, refs[0]) if p):
                continue
        parts = Path(image_path).parts
        image_id = "/".join(parts[-4:]).rsplit(".", 1)[0]
        mask = entry.get("mask_path")
        questions = [
            Question(
                id=f"{image_id}#{i}",
                text=q["Question"],
                options=_parse_options(q["Options"]),
                answer=str(q["Answer"]).strip().upper()[:1] if q.get("Answer") else None,
                subtask=parse_subtask(q["type"]),
            )
            for i, q in enumerate(entry["conversation"])
        ]
        samples.append(
            Sample(
                image_id=image_id,
                image_path=image_path,
                mask_path=mask,
                source=parts[0] if parts else "unknown",
                category=parts[1] if len(parts) > 1 else "unknown",
                is_defective=bool(mask),
                templates=list(entry["random_templates"]),
                questions=questions,
            )
        )
        if limit and len(samples) >= limit:
            break
    return samples

In [ ]:
%%writefile inspector/schemas/state.py
"""Shared workflow state, one instance per image, passed through every pass."""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel, Field

from inspector.schemas.agent_outputs import CriticOutput, FinalAnswer, ReaderOutput, VisualExpertOutput
from inspector.schemas.mmad import Question, Sample


class InspectionState(BaseModel):
    image_id: str
    query_image_path: str
    reference_image_path: str            # == random_templates[0]
    questions: list[Question]

    visual_evidence: VisualExpertOutput | None = None
    reader_a_responses: dict[str, ReaderOutput | None] = Field(default_factory=dict)  # None = invalid
    reader_b_responses: dict[str, ReaderOutput | None] = Field(default_factory=dict)
    critic_verdict: CriticOutput | None = None
    final_answers: dict[str, FinalAnswer] = Field(default_factory=dict)
    telemetry_logs: list[dict[str, Any]] = Field(default_factory=list)

    reopen_count: int = 0                # guard: never exceeds settings.max_reopen

    @classmethod
    def from_sample(cls, s: Sample, image_root: str = "") -> "InspectionState":
        from pathlib import Path

        root = Path(image_root) if image_root else None
        resolve = (lambda p: str(root / p)) if root else (lambda p: p)
        return cls(
            image_id=s.image_id,
            query_image_path=resolve(s.image_path),
            reference_image_path=resolve(s.reference),
            questions=s.questions,
        )

In [ ]:
%%writefile inspector/utils/metrics.py
"""Evaluation: per-subtask accuracy, balanced accuracy, FPR / miss rate, McNemar, bootstrap CI.

A *row* is one scored question: ``{image_id, subtask, correct, is_defective, source}``.
Unanswered questions (``pred is None``) are rows with ``correct=False`` (they stay in the denominator).
"""
from __future__ import annotations

import random
from collections import defaultdict
from typing import Callable, Iterable

from scipy.stats import binomtest

from inspector.schemas.mmad import Sample, SubTask
from inspector.schemas.state import InspectionState

Row = dict


def build_rows(samples: Iterable[Sample], states: Iterable[InspectionState]) -> list[Row]:
    by_id = {s.image_id: s for s in samples}
    rows: list[Row] = []
    for st in states:
        sample = by_id[st.image_id]
        for q in sample.questions:
            fa = st.final_answers.get(q.id)
            pred = fa.selected_option if fa else None
            rows.append({
                "image_id": st.image_id, "question_id": q.id, "subtask": q.subtask,
                "correct": pred is not None and pred == q.answer,
                "is_defective": sample.is_defective, "source": sample.source,
            })
    return rows


def per_subtask_accuracy(rows: list[Row]) -> dict[SubTask, float]:
    """Accuracy per subtask; discrimination is *balanced* (mean of normal and defective accuracy)."""
    acc: dict[SubTask, float] = {}
    for st in SubTask:
        sub = [r for r in rows if r["subtask"] == st]
        if not sub:
            continue
        if st == SubTask.ANOMALY_DISCRIMINATION:
            acc[st] = balanced_accuracy(sub)
        else:
            acc[st] = sum(r["correct"] for r in sub) / len(sub)
    return acc


def mean_accuracy(rows: list[Row]) -> float:
    """Official metric: simple mean over the subtasks present."""
    acc = per_subtask_accuracy(rows)
    return sum(acc.values()) / len(acc) if acc else float("nan")


def balanced_accuracy(disc_rows: list[Row]) -> float:
    groups = [[r["correct"] for r in disc_rows if r["is_defective"] == d] for d in (False, True)]
    rates = [sum(g) / len(g) for g in groups if g]
    return sum(rates) / len(rates) if rates else float("nan")


def fpr_and_miss(disc_rows: list[Row]) -> tuple[float, float]:
    """(false-positive rate on normal parts, missed-defect rate on defective parts)."""
    normal = [r for r in disc_rows if not r["is_defective"]]
    bad = [r for r in disc_rows if r["is_defective"]]
    fpr = sum(not r["correct"] for r in normal) / len(normal) if normal else float("nan")
    miss = sum(not r["correct"] for r in bad) / len(bad) if bad else float("nan")
    return fpr, miss


def mcnemar_exact(rows_a: list[Row], rows_b: list[Row]) -> tuple[int, int, float]:
    """Exact (binomial) two-sided McNemar on paired questions -> (a_only, b_only, p_value)."""
    ca = {r["question_id"]: r["correct"] for r in rows_a}
    cb = {r["question_id"]: r["correct"] for r in rows_b}
    ids = ca.keys() & cb.keys()
    a_only = sum(ca[i] and not cb[i] for i in ids)
    b_only = sum(cb[i] and not ca[i] for i in ids)
    n = a_only + b_only
    return a_only, b_only, 1.0 if n == 0 else float(binomtest(min(a_only, b_only), n, 0.5).pvalue)


def bootstrap_ci(rows: list[Row], stat: Callable[[list[Row]], float] = mean_accuracy,
                 n_boot: int = 1000, seed: int = 0, alpha: float = 0.05) -> tuple[float, float]:
    """Image-level percentile bootstrap (resample images, not questions: questions are correlated)."""
    by_img: dict[str, list[Row]] = defaultdict(list)
    for r in rows:
        by_img[r["image_id"]].append(r)
    ids, rng = list(by_img), random.Random(seed)
    stats = sorted(
        stat([r for i in (rng.choice(ids) for _ in ids) for r in by_img[i]]) for _ in range(n_boot)
    )
    return stats[int(alpha / 2 * n_boot)], stats[min(n_boot - 1, int((1 - alpha / 2) * n_boot))]

In [ ]:
%%writefile inspector/utils/splits.py
"""Stratified dev/test split (fixed seed). MMAD is test-only, so we carve out a dev set for prompts."""
from __future__ import annotations

import json
import random
from collections import defaultdict
from pathlib import Path

from inspector.schemas.mmad import Sample


def make_split(samples: list[Sample], dev_fraction: float, seed: int) -> tuple[list[Sample], list[Sample]]:
    """Stratify by (source, category, is_defective); every stratum with >= 2 images gets >= 1 dev image."""
    strata: dict[tuple, list[Sample]] = defaultdict(list)
    for s in samples:
        strata[(s.source, s.category, s.is_defective)].append(s)
    rng = random.Random(seed)
    dev: list[Sample] = []
    test: list[Sample] = []
    for key in sorted(strata, key=str):
        group = sorted(strata[key], key=lambda s: s.image_id)
        rng.shuffle(group)
        k = round(len(group) * dev_fraction)
        k = max(1, k) if len(group) >= 2 else 0
        dev += group[:k]
        test += group[k:]
    return dev, test


def save_split(dev: list[Sample], test: list[Sample], out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "dev_ids.json").write_text(json.dumps([s.image_id for s in dev]), encoding="utf-8")
    (out_dir / "test_ids.json").write_text(json.dumps([s.image_id for s in test]), encoding="utf-8")

In [ ]:
%%writefile inspector/utils/telemetry.py
"""Structured telemetry: JSONL per run (always on) + optional Logfire/OpenTelemetry.

Each ``span`` writes one JSON line with: run, image_id, agent, prompt(s), raw response(s),
latency (ms), GPU memory (MB), tokens in/out and any extra fields the agent adds
(e.g. the final verdict).
"""
from __future__ import annotations

import json
import threading
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Iterator


def _gpu_mem_mb() -> float | None:
    try:
        import torch

        if torch.cuda.is_available():
            return round(torch.cuda.max_memory_allocated() / 2**20, 1)
    except Exception:  # torch missing or CUDA broken: telemetry must never crash a run
        pass
    return None


class Telemetry:
    def __init__(self, jsonl_path: str | Path, run: str = "run", use_logfire: bool = False):
        self.path = Path(jsonl_path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.run = run
        self._lock = threading.Lock()
        self._logfire: Any = None
        if use_logfire:
            try:  # optional dependency; no account needed unless this flag is set
                import logfire

                logfire.configure(send_to_logfire="if-token-present")
                logfire.instrument_pydantic_ai()  # traces every agent run, as in PasteurAIze
                self._logfire = logfire
            except Exception as e:  # noqa: BLE001
                self.write({"event": "logfire_unavailable", "error": repr(e)})

    def write(self, record: dict[str, Any]) -> None:
        record = {"run": self.run, "ts": time.time(), **record}
        with self._lock, self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

    @contextmanager
    def span(self, agent: str, image_id: str) -> Iterator[dict[str, Any]]:
        """Yield a dict the agent fills (``prompt``, ``raw``, ``tokens_in``, ``tokens_out``...).

        The record is written even when the agent raises (with ``error``), then re-raised.
        """
        rec: dict[str, Any] = {"agent": agent, "image_id": image_id}
        cm = self._logfire.span("{agent} {image_id}", agent=agent, image_id=image_id) if self._logfire else None
        if cm:
            cm.__enter__()
        t0 = time.perf_counter()
        try:
            yield rec
        except Exception as e:
            rec["error"] = repr(e)
            raise
        finally:
            rec["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2)
            rec["gpu_mem_mb"] = _gpu_mem_mb()
            self.write(rec)
            if cm:
                cm.__exit__(None, None, None)

### Test de connexion
Un appel au lecteur A, initialisé exactement comme dans le pipeline (réglages de `Settings`).

In [ ]:
from pydantic_ai import Agent, ImageUrl
from inspector.agents.llm import make_role_model
from inspector.config.settings import get_settings

settings = get_settings()
# Même initialisation que les agents du pipeline : OpenAIChatModel(model_name, OpenAIProvider(api_key, base_url)),
# avec le repli Gemini (FallbackModel) si PROVIDER = "groq" et qu'une clé Gemini est fournie.
ping_model = make_role_model(settings, settings.reader_a_model, settings.remote.get("reader_a"))
ping_agent = Agent(model=ping_model, name="Ping Agent", system_prompt="You are a concise visual assistant.", retries=3)
r = await ping_agent.run(["Describe this image in one sentence.", ImageUrl(url="https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")])
print(r.all_messages()[-1].model_name, "->", r.output)

## 4. Tests (sans GPU)

In [ ]:
%%writefile tests/test_workflow.py
"""GPU-free tests: schemas, strict 1-shot, critic/reopen guard, aggregator, metrics, pass runner."""
import json

import pytest
from pydantic import ValidationError
from pydantic_ai import models
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel

from inspector.agents.aggregator import Aggregator
from inspector.agents.critic_verifier import CriticVerifier, cell_in_text
from inspector.agents.llm import make_model, make_role_model
from inspector.agents.mock import MockExpert
from inspector.agents.reader_mllm import ReaderMLLM
from inspector.config.settings import Settings
from inspector.orchestrator.graph import Step, build_steps
from inspector.orchestrator.pass_runner import PassRunner
from inspector.schemas.agent_outputs import ReaderOutput, VisualExpertOutput
from inspector.schemas.mmad import Question, Sample, SubTask, load_mmad
from inspector.schemas.state import InspectionState
from inspector.utils import metrics
from inspector.utils.splits import make_split
from inspector.utils.telemetry import Telemetry


@pytest.fixture(autouse=True)
def _clean_env(monkeypatch):
    """Settings reads os.environ: ignore ONEREF_* / keys set by a notebook profile or the shell."""
    import os
    for k in [k for k in os.environ if k.startswith("ONEREF_")] + ["HF_TOKEN", "XAI_API_KEY"]:
        monkeypatch.delenv(k, raising=False)

models.ALLOW_MODEL_REQUESTS = False  # tests never call a real LLM endpoint

DISC = {"A": "Yes", "B": "No"}
LOC = {"A": "top left", "B": "top right", "C": "center", "D": "bottom right"}


def mk_questions(img="MVTec/bottle/x"):
    return [
        Question(id=f"{img}#0", text="Is there any defect?", options=DISC, answer="A", subtask=SubTask.ANOMALY_DISCRIMINATION),
        Question(id=f"{img}#1", text="Where is it?", options=LOC, answer="B", subtask=SubTask.DEFECT_LOCALIZATION),
    ]


def mk_sample(i=0, defective=True):
    img = f"MVTec/bottle/{i}"
    return Sample(image_id=img, image_path=img + ".png", is_defective=defective,
                  templates=["ref0.png", "ref1.png"], questions=mk_questions(img))


def mk_state(i=0):
    return InspectionState.from_sample(mk_sample(i))


def ans(qid, letter, conf=0.9):
    return ReaderOutput(question_id=qid, selected_option=letter, confidence=conf)


def expert(score=0.9, region="top right"):
    return VisualExpertOutput(heatmap_path="h", candidate_region=region, confidence_score=score, crop_path="c")


# ---- schemas -----------------------------------------------------------------
def test_schema_rejects_bad_values():
    with pytest.raises(ValidationError):
        ReaderOutput(question_id="q", selected_option="E", confidence=0.5)
    with pytest.raises(ValidationError):
        ReaderOutput(question_id="q", selected_option="A", confidence=1.5)
    with pytest.raises(ValidationError):
        VisualExpertOutput(heatmap_path="h", candidate_region="nowhere", confidence_score=0.5, crop_path="c")
    with pytest.raises(ValidationError):
        Question(id="q", text="t", options=DISC, answer="C", subtask=SubTask.ANOMALY_DISCRIMINATION)


def test_strict_one_shot_reference():
    assert mk_sample().reference == "ref0.png"
    assert mk_state().reference_image_path == "ref0.png"


def test_load_mmad_ignores_similar_templates(tmp_path):
    entry = {"conversation": [{"Question": "Is there any defect?", "Options": DISC, "Answer": "B", "type": "Anomaly Discrimination"}],
             "random_templates": ["r0.png", "r1.png"], "similar_templates": ["s0.png"], "mask_path": None}
    p = tmp_path / "mmad.json"
    p.write_text(json.dumps({"MVTec/bottle/test/good/000.png": entry}))
    (s,) = load_mmad(p)
    assert s.reference == "r0.png" and not hasattr(s, "similar_templates") and not s.is_defective


def scripted(*replies):
    """FunctionModel replying with the given texts / tool calls in order; records the calls."""
    calls = []

    def fn(messages, info):
        calls.append(messages)
        r = replies[min(len(calls), len(replies)) - 1]
        part = ToolCallPart(tool_name=r[1], args={}) if isinstance(r, tuple) else TextPart(r)
        return ModelResponse(parts=[part])

    return FunctionModel(fn), calls


def run_reader(model, state):
    r = ReaderMLLM(Settings(json_retries=1), "m", "reader_a", model=model)
    r.image_parts = lambda st: []
    r.load()
    try:
        return r._run(state, rec := {}), rec
    finally:
        r.unload()


def test_reader_agent_retries_on_option_not_offered():
    st = mk_state()
    st.questions = st.questions[:1]
    # "C" is not offered on a yes/no question -> output_validator raises ModelRetry -> second answer kept
    model, calls = scripted('{"selected_option": "C", "confidence": 0.7}', '{"selected_option": "B", "confidence": 0.7}')
    st, rec = run_reader(model, st)
    a = st.reader_a_responses[st.questions[0].id]
    assert a.selected_option == "B" and a.question_id == st.questions[0].id
    assert len(calls) == 2 and rec["calls"][0]["raw"] and "tokens_in" in rec


def test_reader_agent_gives_none_after_retries():
    model, _ = scripted("not json at all")
    st, rec = run_reader(model, mk_state())
    assert all(v is None for v in st.reader_a_responses.values())
    assert all("error" in c for c in rec["calls"])


# ---- critic ------------------------------------------------------------------
def test_critic_conflict_reopens_once():
    s = Settings()
    st = mk_state()
    st.visual_evidence = expert(0.9, "top right")
    q0, q1 = st.questions
    st.reader_a_responses = {q0.id: ans(q0.id, "B"), q1.id: ans(q1.id, "A")}  # "No defect" + wrong cell
    st.reader_b_responses = {q0.id: ans(q0.id, "A"), q1.id: ans(q1.id, "B")}
    critic = CriticVerifier(s)
    st = critic._run(st, {})
    assert st.critic_verdict.conflict_detected and set(st.critic_verdict.reopen_questions) == {q0.id, q1.id}
    assert st.reopen_count == 1
    st = critic._run(st, {})  # a second critic run must NOT grant another reopen
    assert st.reopen_count == 1 and st.critic_verdict.reopen_questions == []


def test_critic_flags_missing_answer_as_invalid():
    st = mk_state()
    st.reader_a_responses = {q.id: None for q in st.questions}
    st.reader_b_responses = {q.id: ans(q.id, "A") for q in st.questions}
    st = CriticVerifier(Settings())._run(st, {})
    assert not st.critic_verdict.is_valid


def test_llm_critic_calls_rule_tool_rejects_unknown_ids_and_merges_with_rules():
    st = mk_state()
    st.visual_evidence = expert(0.9, "top right")
    q0, q1 = st.questions
    st.reader_a_responses = {q0.id: ans(q0.id, "B"), q1.id: ans(q1.id, "B")}  # rules: q0 contradicts expert
    st.reader_b_responses = {q0.id: ans(q0.id, "A"), q1.id: ans(q1.id, "B")}
    bad = '{"is_valid": true, "conflict_detected": true, "reopen_questions": ["nope"], "critique_reason": "x"}'
    good = f'{{"is_valid": true, "conflict_detected": true, "reopen_questions": ["{q1.id}"], "critique_reason": "loc looks wrong"}}'
    model, calls = scripted(("tool", "run_consistency_rules"), bad, good)
    critic = CriticVerifier(Settings(json_retries=1), model=model)
    critic.image_parts = lambda s: []
    critic.load()
    try:
        st = critic._run(st, rec := {})
    finally:
        critic.unload()
    tool_out = [p.content for m in calls[1] for p in getattr(m, "parts", []) if isinstance(p, ToolReturnPart)]
    assert "said no defect" in tool_out[0]                       # the rules reached the LLM via the tool
    assert set(st.critic_verdict.reopen_questions) == {q0.id, q1.id}  # rules ∪ LLM, unknown id retried away
    assert "reviewer: loc looks wrong" in st.critic_verdict.critique_reason


def test_llm_critic_failure_keeps_rules_verdict():
    st = mk_state()
    st.visual_evidence = expert(0.9, "top right")
    q0, q1 = st.questions
    st.reader_a_responses = st.reader_b_responses = {q0.id: ans(q0.id, "B"), q1.id: ans(q1.id, "B")}
    model, _ = scripted("garbage")
    critic = CriticVerifier(Settings(), model=model)
    critic.image_parts = lambda s: []
    critic.load()
    st = critic._run(st, rec := {})
    critic.unload()
    assert st.critic_verdict.reopen_questions == [q0.id] and "llm_error" in rec


def test_cell_in_text_longest_match():
    assert cell_in_text("top-center area") == "top center" and cell_in_text("in the center") == "center"


# ---- aggregator --------------------------------------------------------------
def test_aggregator_weighted_vote_and_tiebreak():
    s = Settings(weights={"reader_a": {"defect_localization": 2.0}})
    st = mk_state()
    q0, q1 = st.questions
    st.reader_a_responses = {q0.id: ans(q0.id, "A", 0.6), q1.id: ans(q1.id, "A")}
    st.reader_b_responses = {q0.id: ans(q0.id, "B", 0.9), q1.id: ans(q1.id, "B")}
    st = Aggregator(s)._run(st, {})
    assert st.final_answers[q1.id].selected_option == "A"   # weight 2 beats 1
    assert st.final_answers[q0.id].selected_option == "B"   # tie 1-1 -> higher confidence


def test_aggregator_expert_votes_on_discrimination_and_none_if_no_votes():
    st = mk_state()
    q0, q1 = st.questions
    st.visual_evidence = expert(0.9)
    st.reader_a_responses = {q0.id: ans(q0.id, "B"), q1.id: None}
    st.reader_b_responses = {q0.id: None, q1.id: None}
    st = Aggregator(Settings(expert_weight_discrimination=2.0))._run(st, {})
    assert st.final_answers[q0.id].selected_option == "A"  # expert "Yes" (weight 2) outvotes reader "No" (1)
    assert st.final_answers[q1.id].selected_option is None


# ---- metrics -----------------------------------------------------------------
def rows(spec):
    return [{"image_id": i, "question_id": f"{i}#0", "subtask": SubTask.ANOMALY_DISCRIMINATION,
             "correct": c, "is_defective": d, "source": "x"} for i, (c, d) in enumerate(spec)]


def test_balanced_accuracy_and_fpr():
    r = rows([(True, False), (True, False), (True, False), (False, True)])  # 3/3 normal, 0/1 defective
    assert metrics.balanced_accuracy(r) == pytest.approx(0.5)
    assert metrics.fpr_and_miss(r) == (0.0, 1.0)


def test_mcnemar_and_bootstrap():
    a = rows([(True, True)] * 20 + [(False, True)] * 0)
    b = rows([(False, True)] * 20)
    n_a, n_b, p = metrics.mcnemar_exact(a, b)
    assert (n_a, n_b) == (20, 0) and p < 1e-4
    assert metrics.mcnemar_exact(a, a)[2] == 1.0
    lo, hi = metrics.bootstrap_ci(rows([(True, True), (False, False)] * 10), n_boot=100)
    assert 0 <= lo <= hi <= 1


def test_split_is_stratified_and_deterministic():
    samples = [mk_sample(i, defective=i % 2 == 0) for i in range(20)]
    d1, t1 = make_split(samples, 0.1, seed=0)
    d2, _ = make_split(samples, 0.1, seed=0)
    assert [s.image_id for s in d1] == [s.image_id for s in d2]
    assert len(d1) + len(t1) == 20 and not {s.image_id for s in d1} & {s.image_id for s in t1}
    assert {s.is_defective for s in d1} == {True, False}


# ---- pass runner -------------------------------------------------------------
class Spy(MockExpert):
    events: list = []

    def load(self):
        Spy.events.append("load")

    def unload(self):
        Spy.events.append("unload")
        super().unload()


def test_pass_runner_unloads_each_pass_and_resumes_from_cache(tmp_path):
    Spy.events = []
    tel = Telemetry(tmp_path / "t.jsonl")
    steps = [Step("1_expert", Spy), Step("2_expert_again", Spy)]
    runner = PassRunner(tmp_path, tel)
    states = [mk_state(i) for i in range(3)]
    out = runner.run(steps, states)
    assert Spy.events == ["load", "unload", "load", "unload"]  # one model at a time
    assert all(s.visual_evidence for s in out)
    Spy.events = []
    runner.run(steps, [mk_state(i) for i in range(3)])  # all cached -> nothing loaded
    assert Spy.events == []
    assert any("latency_ms" in json.loads(l) for l in (tmp_path / "t.jsonl").read_text().splitlines())


def test_full_mock_pipeline_end_to_end(tmp_path):
    s = Settings()
    tel = Telemetry(tmp_path / "t.jsonl")
    samples = [mk_sample(i) for i in range(4)]
    states = [InspectionState.from_sample(x) for x in samples]
    out = PassRunner(tmp_path, tel).run(build_steps(s, tmp_path, mock=True), states)
    assert all(len(st.final_answers) == 2 for st in out)
    assert all(st.reopen_count <= s.max_reopen for st in out)
    r = metrics.build_rows(samples, out)
    assert 0 <= metrics.mean_accuracy(r) <= 1


# ---- visual expert (pure functions; model loading needs weights, not tested here) ----
def test_expert_anomaly_map_and_grid():
    import numpy as np
    from inspector.agents.visual_expert import anomaly_map, grid_cell
    r = np.eye(4, dtype=np.float32)
    q = np.vstack([r[:3], [[1, 1, 1, 1]]]).astype(np.float32)  # 3 patches match reference, last one differs
    d = anomaly_map(q, r)
    assert d[:3].max() < 1e-6 and d[3] > 0.4
    assert grid_cell(0, 0, 90, 90) == "top left" and grid_cell(89, 89, 90, 90) == "bottom right" and grid_cell(45, 45, 90, 90) == "center"


def test_reader_fails_fast_without_llm_server():
    r = ReaderMLLM(Settings(_env_file=None, llm_provider="vllm", llm_base_url="http://127.0.0.1:9/v1", vllm_autostart=False), "Qwen/Qwen3-VL-8B-Instruct", "reader_a")
    with pytest.raises(RuntimeError, match="No LLM server"):
        r.load()


def test_remote_role_uses_its_own_endpoint_and_no_local_server(monkeypatch):
    import inspector.agents.llm as llm

    monkeypatch.setattr(llm.VLLMServer, "start", lambda self: pytest.fail("local server must not start"))
    s = Settings(_env_file=None, backup_enabled=False, remote={"reader_a": {"base_url": "https://api.x.ai/v1", "api_key": "k", "model": "grok-x"}})
    r = ReaderMLLM(s, "Qwen/Qwen3-VL-8B-Instruct", "reader_a_reopen", reopen=True)
    r.load()
    try:
        assert r.agent.model.model_name == "grok-x" and "api.x.ai" in str(r.agent.model.client.base_url)
    finally:
        r.unload()


def test_grok_fills_remote_for_three_roles():
    s = Settings(_env_file=None, use_grok=True, grok_api_key="k")
    assert set(s.remote) == {"reader_a", "reader_b", "critic"}
    assert all("api.x.ai" in ep["base_url"] and ep["api_key"] == "k" for ep in s.remote.values())
    assert "api.x.ai" in make_model(s, "x", s.remote["critic"]).client.base_url.host


def test_grok_without_key_is_an_error(monkeypatch):
    monkeypatch.delenv("XAI_API_KEY", raising=False)
    with pytest.raises(ValueError, match="GROK_API_KEY"):
        Settings(_env_file=None, use_grok=True)


def test_router_is_default_and_starts_no_server(monkeypatch):
    import inspector.agents.llm as llm

    monkeypatch.setattr(llm.VLLMServer, "start", lambda self: pytest.fail("router must not start a server"))
    monkeypatch.setenv("HF_TOKEN", "hf_test")
    s = Settings(_env_file=None)  # ignore the local .env: this checks the built-in defaults
    r = ReaderMLLM(s, s.reader_a_model, "reader_a")
    r.load()
    try:
        assert "router.huggingface.co" in str(r.agent.model.client.base_url) and r.agent.model.client.api_key == "hf_test"
    finally:
        r.unload()


def test_remote_role_gets_a_backup_endpoint(monkeypatch):
    from pydantic_ai.models.fallback import FallbackModel
    s = Settings(_env_file=None, llm_api_key="gem-key", use_grok=True, grok_api_key="groq-key")
    assert isinstance(make_role_model(s, "gemini-x", s.remote["reader_a"]), FallbackModel)
    assert not isinstance(make_role_model(s, "gemini-x"), FallbackModel)            # no remote: nothing to back up
    off = Settings(_env_file=None, llm_api_key="gem-key", use_grok=True, grok_api_key="groq-key", backup_enabled=False)
    assert not isinstance(make_role_model(off, "gemini-x", off.remote["reader_a"]), FallbackModel)
    nokey = Settings(_env_file=None, use_grok=True, grok_api_key="groq-key")        # llm_api_key stays "EMPTY"
    monkeypatch.delenv("HF_TOKEN", raising=False)
    assert not isinstance(make_role_model(nokey, "gemini-x", nokey.remote["reader_a"]), FallbackModel)


def test_backup_answers_when_primary_hits_quota():
    from pydantic_ai import Agent
    from pydantic_ai.exceptions import ModelHTTPError
    from pydantic_ai.models.fallback import FallbackModel

    def quota(messages, info):
        raise ModelHTTPError(429, "groq", {"error": "rate limit"})

    def ok(messages, info):
        return ModelResponse(parts=[TextPart("from backup")])

    agent = Agent(FallbackModel(FunctionModel(quota), FunctionModel(ok)))
    assert agent.run_sync("hi").output == "from backup"


def test_load_mmad_available_under_keeps_only_existing_images(tmp_path):
    entry = lambda img, ref: {"random_templates": [ref], "conversation": [
        {"Question": "q?", "Options": {"A": "a", "B": "b"}, "Answer": "A", "type": "Anomaly Discrimination"}]}
    data = {"S/c/test/good/1.png": entry(0, "S/c/train/good/0.png"), "S/c/test/good/2.png": entry(0, "S/c/train/good/0.png")}
    (tmp_path / "mmad.json").write_text(json.dumps(data))
    for p in ("S/c/test/good/1.png", "S/c/train/good/0.png"):
        (tmp_path / p).parent.mkdir(parents=True, exist_ok=True)
        (tmp_path / p).write_bytes(b"x")
    assert len(load_mmad(tmp_path / "mmad.json")) == 2
    kept = load_mmad(tmp_path / "mmad.json", available_under=tmp_path)
    assert [x.image_path for x in kept] == ["S/c/test/good/1.png"]

In [ ]:
!python -m pytest -q tests -p no:cacheprovider

## 5. Exécution

In [ ]:
# Smoke test sans GPU (agents factices)
!python -m inspector.main --split dev --mock --limit 20 --run-name smoke {OPT}

In [ ]:
# Sonde : 50 images
!python -m inspector.main --split dev --limit 50 --run-name probe {OPT}

In [ ]:
# Run dev complet (reprenable : relancer après une coupure)
!python -m inspector.main --split dev --run-name dev01 {OPT}

In [ ]:
# Dernières lignes du log JSONL (portable Windows / Linux)
from pathlib import Path
for f in Path("outputs/dev01").glob("*.jsonl"):
    print(f); [print(l[:600]) for l in f.read_text(encoding="utf-8").splitlines()[-3:]]

## Test (une seule fois, config figée)
Ne jamais lancer le split test pour régler des paramètres.

In [ ]:
# !python -m inspector.main --split test --frozen-config --run-name test01 {OPT}